# DeepGPでの関数Fitting性能評価: 2次元関数 f₁(x) = ||z||² + f_opt

本notebookでは、画像に示された2次元関数の近似性能をDeepGPで評価します。

## 目標関数
- $f_1(\mathbf{x}) = \|\mathbf{z}\|^2 + f_{\text{opt}}$
- $\mathbf{z} = \mathbf{x} - \mathbf{x}^{\text{opt}}$
- $\mathbf{x}^{\text{opt}} = [0.5, 0.3]$ (固定値)
- $f_{\text{opt}} = -10.0$ (固定値)

## 1. ライブラリのインポートと初期設定

In [ ]:
# 初期設定: 必要なライブラリをインポートし、パスを設定
import os
import random
import sys
import time

sys.path.append("../../")
import gpytorch
import japanize_matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import tqdm.notebook
from gpytorch.distributions import MultivariateNormal
from gpytorch.kernels import RBFKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.mlls import DeepApproximateMLL, VariationalELBO
from gpytorch.models.deep_gps import DeepGP, DeepGPLayer
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy
from torch.utils.data import DataLoader, TensorDataset


# 再現性のための乱数シード設定
def set_seed(seed=42):
    """完全な再現性のための乱数シード設定"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def worker_init_fn(worker_id):
    """DataLoaderワーカープロセス用の乱数シード設定"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(42)  # 全体の乱数シード設定
print("ライブラリのインポートと初期設定完了")

## 2. 目標関数の定義とデータセット作成

In [ ]:
# 目標関数の定義: f₁(x) = ||z||² + f_opt
# z = x - x_opt

# 固定パラメータ
x_opt = np.array([0.5, 0.3])  # 最適解位置
f_opt = -10.0  # 最適値


def target_function(x):
    """
    目標関数: f₁(x) = ||z||² + f_opt
    Args:
        x: 入力ベクトル (N, 2) または (2,)
    Returns:
        関数値 (N,) または スカラー
    """
    x = np.array(x)
    if x.ndim == 1:
        x = x.reshape(1, -1)

    # z = x - x_opt
    z = x - x_opt.reshape(1, -1)

    # ||z||²の計算
    z_squared_norm = np.sum(z**2, axis=1)

    # f₁(x) = ||z||² + f_opt
    result = z_squared_norm + f_opt

    return result if len(result) > 1 else result[0]


# 動作確認
print("目標関数のテスト:")
print(f"x_opt = {x_opt}")
print(f"f_opt = {f_opt}")
print(f"f(x_opt) = {target_function(x_opt):.6f} (should be {f_opt})")
print(f"f([0, 0]) = {target_function([0, 0]):.6f}")
print(f"f([1, 1]) = {target_function([1, 1]):.6f}")

In [ ]:
# データセットの作成
num_samples = 1000
input_range = [-2, 2]  # 入力範囲

# 2次元入力点をランダムサンプリング
X = np.random.uniform(input_range[0], input_range[1], (num_samples, 2))

# 目標関数値を計算
y = target_function(X)

# PyTorchテンソルに変換
X_tensor = torch.FloatTensor(X)
y_tensor = torch.FloatTensor(y).reshape(-1, 1)

# データ分割（学習:検証:テスト = 7:1.5:1.5）
train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train = X_tensor[:train_size]
y_train = y_tensor[:train_size]

X_val = X_tensor[train_size : train_size + val_size]
y_val = y_tensor[train_size : train_size + val_size]

X_test = X_tensor[train_size + val_size :]
y_test = y_tensor[train_size + val_size :]

# データセット作成
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# データローダー作成
batch_size = 64  # DeepGPでは少し大きめのバッチサイズを使用
generator = torch.Generator()
generator.manual_seed(42)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    worker_init_fn=worker_init_fn,
    generator=generator,
)

print(f"データセット作成完了:")
print(f"  学習データ: {len(train_dataset)} samples")
print(f"  検証データ: {len(val_dataset)} samples")
print(f"  テストデータ: {len(test_dataset)} samples")
print(f"  入力次元: {X_train.shape[1]}")
print(f"  関数値範囲: [{y.min():.3f}, {y.max():.3f}]")
print(f"  バッチサイズ: {batch_size}")

In [ ]:
# 関数の可視化
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 3D表面プロット用のグリッド作成
x1_grid = np.linspace(input_range[0], input_range[1], 50)
x2_grid = np.linspace(input_range[0], input_range[1], 50)
X1_grid, X2_grid = np.meshgrid(x1_grid, x2_grid)
grid_points = np.column_stack([X1_grid.ravel(), X2_grid.ravel()])
Z_grid = target_function(grid_points).reshape(X1_grid.shape)

# 左: 等高線プロット
contour = axes[0].contour(X1_grid, X2_grid, Z_grid, levels=20, colors="blue", alpha=0.6)
axes[0].clabel(contour, inline=True, fontsize=8)
axes[0].scatter(
    X_train[:100, 0],
    X_train[:100, 1],
    c=y_train[:100],
    cmap="viridis",
    s=20,
    alpha=0.7,
    label="Training Data (subset)",
)
axes[0].scatter(
    x_opt[0],
    x_opt[1],
    c="red",
    s=100,
    marker="*",
    label=f"Optimum ({x_opt[0]}, {x_opt[1]})",
    zorder=5,
)
axes[0].set_xlabel("x₁")
axes[0].set_ylabel("x₂")
axes[0].set_title("Target Function: f₁(x) = ||z||² + f_opt (Contour)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右: 散布図（関数値の分布）
scatter = axes[1].scatter(
    X_train[:, 0], X_train[:, 1], c=y_train.numpy(), cmap="viridis", s=20, alpha=0.7
)
axes[1].scatter(
    x_opt[0],
    x_opt[1],
    c="red",
    s=100,
    marker="*",
    label=f"Optimum (f={f_opt})",
    zorder=5,
)
plt.colorbar(scatter, ax=axes[1], label="Function Value")
axes[1].set_xlabel("x₁")
axes[1].set_ylabel("x₂")
axes[1].set_title("Training Data Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"最適解での関数値: f(x_opt) = {target_function(x_opt):.6f}")
print(f"データの統計:")
print(f"  平均: {y.mean():.3f}")
print(f"  標準偏差: {y.std():.3f}")
print(f"  最小値: {y.min():.3f}")
print(f"  最大値: {y.max():.3f}")

## 3. DeepGPモデルの実装（2次元入力対応）

In [ ]:
# DeepGPのレイヤー定義（24番ノートブックを2次元入力に対応させたもの）
class ToyDeepGPHiddenLayer2D(DeepGPLayer):
    def __init__(self, input_dims, output_dims, num_inducing=20, mean_type="constant"):
        if output_dims is None:
            inducing_points = torch.randn(num_inducing, input_dims)
            batch_shape = torch.Size([])
        else:
            inducing_points = torch.randn(output_dims, num_inducing, input_dims)
            batch_shape = torch.Size([output_dims])

        variational_distribution = CholeskyVariationalDistribution(
            num_inducing_points=num_inducing, batch_shape=batch_shape
        )

        variational_strategy = VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )

        super(ToyDeepGPHiddenLayer2D, self).__init__(
            variational_strategy, input_dims, output_dims
        )

        if mean_type == "constant":
            self.mean_module = ConstantMean(batch_shape=batch_shape)
        else:
            self.mean_module = LinearMean(input_dims)

        # ARDカーネルを使用（各次元で異なるlengthscale）
        self.covar_module = ScaleKernel(
            RBFKernel(batch_shape=batch_shape, ard_num_dims=input_dims),
            batch_shape=batch_shape,
            ard_num_dims=None,
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultivariateNormal(mean_x, covar_x)

    def __call__(self, x, *other_inputs, **kwargs):
        """
        Skip connection対応の__call__メソッド
        """
        if len(other_inputs):
            if isinstance(x, gpytorch.distributions.MultitaskMultivariateNormal):
                x = x.rsample()

            processed_inputs = [
                inp.unsqueeze(0).expand(
                    gpytorch.settings.num_likelihood_samples.value(), *inp.shape
                )
                for inp in other_inputs
            ]

            x = torch.cat([x] + processed_inputs, dim=-1)

        return super().__call__(x, are_samples=bool(len(other_inputs)))


print("DeepGPレイヤークラス定義完了")

In [ ]:
# DeepGPモデル定義（2次元入力対応）
class DeepGPModel2D(DeepGP):
    def __init__(self, train_x_shape, num_hidden_dims=2):
        # 隠れ層（2次元入力 → num_hidden_dims次元出力）
        hidden_layer = ToyDeepGPHiddenLayer2D(
            input_dims=train_x_shape[-1],  # 2次元入力
            output_dims=num_hidden_dims,  # 隠れ層の次元数
            num_inducing=16,  # 誘導点数（4x4相当）
            mean_type="linear",
        )

        # 最終層（隠れ層出力 → 1次元出力）
        last_layer = ToyDeepGPHiddenLayer2D(
            input_dims=hidden_layer.output_dims,
            output_dims=None,  # 1次元出力
            num_inducing=10,  # 最終層の誘導点数
            mean_type="constant",
        )

        super().__init__()

        self.hidden_layer = hidden_layer
        self.last_layer = last_layer
        self.likelihood = GaussianLikelihood()

    def forward(self, inputs):
        hidden_rep = self.hidden_layer(inputs)
        output = self.last_layer(hidden_rep)
        return output

    def predict(self, test_loader):
        """テストデータでの予測"""
        with torch.no_grad():
            mus = []
            variances = []
            lls = []
            for x_batch, y_batch in test_loader:
                with gpytorch.settings.num_likelihood_samples(x_batch.size(0)):
                    preds = self.likelihood(self(x_batch))
                    mus.append(preds.mean)
                    variances.append(preds.variance)
                    # 対数尤度は計算しない（簡単のため）

        return torch.cat(mus, dim=0), torch.cat(variances, dim=0)


# モデルのハイパーパラメータ
num_hidden_dims = 2  # 隠れ層の次元数

# モデル作成
deepgp_model = DeepGPModel2D(X_train.shape, num_hidden_dims=num_hidden_dims)

print(f"DeepGPモデル作成完了:")
print(f"  入力次元: {X_train.shape[-1]}")
print(f"  隠れ層次元: {num_hidden_dims}")
print(f"  出力次元: 1")
print(f"  隠れ層誘導点数: 16")
print(f"  最終層誘導点数: 10")

## 4. モデルの学習

In [ ]:
# DeepGPの学習設定
optimizer = torch.optim.Adam([{"params": deepgp_model.parameters()}], lr=0.001)

# DeepGP用のMLL（変分下界）
mll = DeepApproximateMLL(
    VariationalELBO(deepgp_model.likelihood, deepgp_model, X_train.shape[-2])
)

# 学習ループ
num_epochs = 200
train_losses = []
val_losses = []

print("DeepGP学習開始...")
start_time = time.time()

epochs_iter = tqdm.notebook.tqdm(range(num_epochs), desc="Epoch")
for epoch in epochs_iter:
    # 学習フェーズ
    deepgp_model.train()
    epoch_train_loss = 0.0

    minibatch_iter = tqdm.notebook.tqdm(train_dataloader, desc="Minibatch", leave=False)
    for x_batch, y_batch in minibatch_iter:
        actual_batch_size = x_batch.size(0)

        with gpytorch.settings.num_likelihood_samples(actual_batch_size):
            optimizer.zero_grad()
            output = deepgp_model(x_batch)
            loss = -mll(output, y_batch.squeeze())
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()
            minibatch_iter.set_postfix(loss=loss.item())

    avg_train_loss = epoch_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)

    # 検証フェーズ
    deepgp_model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for x_batch, y_batch in val_dataloader:
            actual_batch_size = x_batch.size(0)
            with gpytorch.settings.num_likelihood_samples(actual_batch_size):
                output = deepgp_model(x_batch)
                loss = -mll(output, y_batch.squeeze())
                epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_dataloader)
    val_losses.append(avg_val_loss)

    # 進捗更新
    epochs_iter.set_postfix(train_loss=avg_train_loss, val_loss=avg_val_loss)

end_time = time.time()
training_time = end_time - start_time

print(f"DeepGP学習完了!")
print(f"学習時間: {training_time:.2f}秒")

# 学習曲線の表示
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Training Loss", alpha=0.7)
plt.plot(val_losses, label="Validation Loss", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Loss (Negative ELBO)")
plt.title("DeepGP Learning Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. モデルの評価

In [ ]:
# 問題の箇所
grid_density = 50
x1_dense = np.linspace(input_range[0], input_range[1], grid_density)
x2_dense = np.linspace(input_range[0], input_range[1], grid_density)
X1_dense, X2_dense = np.meshgrid(x1_dense, x2_dense)
grid_dense = torch.FloatTensor(np.column_stack([X1_dense.ravel(), X2_dense.ravel()]))

print(f"Grid size: {len(grid_dense)}")  # 50 * 50 = 2500

# バッチ処理での問題
dense_batch_size = 100
for i in range(0, len(grid_dense), dense_batch_size):
    batch_grid = grid_dense[i : i + dense_batch_size]
    actual_batch_size = batch_grid.size(0)

    with gpytorch.settings.num_likelihood_samples(actual_batch_size):
        dense_predictions = deepgp_model.likelihood(deepgp_model(batch_grid))
        dense_means.append(dense_predictions.mean)  # ここで異常なサイズが生成

In [ ]:
# 密なグリッドでの予測（修正版）
deepgp_model.eval()
with torch.no_grad():
    dense_means = []
    dense_vars = []

    # バッチサイズを調整
    dense_batch_size = 50  # より小さなバッチサイズ
    total_processed = 0

    for i in range(0, len(grid_dense), dense_batch_size):
        batch_grid = grid_dense[i : i + dense_batch_size]
        actual_batch_size = batch_grid.size(0)

        # サンプリング数を明示的に制御
        with gpytorch.settings.num_likelihood_samples(1):  # 1に固定
            dense_predictions = deepgp_model.likelihood(deepgp_model(batch_grid))

            # 明示的にサイズを確認
            batch_mean = dense_predictions.mean
            batch_var = dense_predictions.variance

            print(
                f"Batch {i//dense_batch_size}: input={actual_batch_size}, output_mean={batch_mean.shape}, output_var={batch_var.shape}"
            )

            # 適切なサイズのみを追加
            if (
                batch_mean.numel() == actual_batch_size
                and batch_var.numel() == actual_batch_size
            ):
                dense_means.append(batch_mean.view(-1))
                dense_vars.append(batch_var.view(-1))
            else:
                print(
                    f"警告: バッチ {i//dense_batch_size} で異常なサイズが検出されました"
                )
                print(
                    f"  Expected: {actual_batch_size}, Got: mean={batch_mean.numel()}, var={batch_var.numel()}"
                )

        total_processed += actual_batch_size

    print(f"Total processed: {total_processed}, Expected: {len(grid_dense)}")

    # 連結前にサイズを確認
    total_predictions = sum(tensor.numel() for tensor in dense_means)
    print(f"Total predictions before concatenation: {total_predictions}")

    if total_predictions == len(grid_dense):
        dense_mean = torch.cat(dense_means, dim=0).reshape(grid_density, grid_density)
        dense_var = torch.cat(dense_vars, dim=0).reshape(grid_density, grid_density)
    else:
        print(
            f"エラー: 予測数が一致しません。Expected: {len(grid_dense)}, Got: {total_predictions}"
        )
        # フォールバック処理
        dense_mean = torch.zeros(grid_density, grid_density)
        dense_var = torch.ones(grid_density, grid_density)

## 6. 結果の可視化

In [ ]:
# テストデータでの予測
deepgp_model.eval()
with torch.no_grad():
    test_means = []
    test_vars = []

    for x_batch, y_batch in test_dataloader:
        # サンプリング数を1に固定
        with gpytorch.settings.num_likelihood_samples(1):
            test_predictions = deepgp_model.likelihood(deepgp_model(x_batch))
            test_means.append(test_predictions.mean.view(-1))
            test_vars.append(test_predictions.variance.view(-1))

    test_mean = torch.cat(test_means, dim=0)
    test_var = torch.cat(test_vars, dim=0)

# 評価指標の計算
test_rmse = torch.sqrt(torch.mean((test_mean - y_test.squeeze()) ** 2))
test_mae = torch.mean(torch.abs(test_mean - y_test.squeeze()))
test_r2 = 1 - torch.sum((y_test.squeeze() - test_mean) ** 2) / torch.sum(
    (y_test.squeeze() - y_test.mean()) ** 2
)

print("=== テスト性能 ===")
print(f"RMSE: {test_rmse.item():.6f}")
print(f"MAE: {test_mae.item():.6f}")
print(f"R²: {test_r2.item():.6f}")
print(f"平均予測分散: {test_var.mean().item():.6f}")

# 密なグリッドでの予測
grid_density = 50
x1_dense = np.linspace(input_range[0], input_range[1], grid_density)
x2_dense = np.linspace(input_range[0], input_range[1], grid_density)
X1_dense, X2_dense = np.meshgrid(x1_dense, x2_dense)
grid_dense = torch.FloatTensor(np.column_stack([X1_dense.ravel(), X2_dense.ravel()]))

# 密なグリッドでの予測（バッチ処理）
with torch.no_grad():
    dense_means = []
    dense_vars = []

    dense_batch_size = 100
    for i in range(0, len(grid_dense), dense_batch_size):
        batch_grid = grid_dense[i : i + dense_batch_size]

        with gpytorch.settings.num_likelihood_samples(1):
            dense_predictions = deepgp_model.likelihood(deepgp_model(batch_grid))
            dense_means.append(dense_predictions.mean.view(-1))
            dense_vars.append(dense_predictions.variance.view(-1))

    dense_mean = torch.cat(dense_means, dim=0).reshape(grid_density, grid_density)
    dense_var = torch.cat(dense_vars, dim=0).reshape(grid_density, grid_density)

# 真の関数値（密なグリッド）
true_dense = target_function(grid_dense.numpy()).reshape(grid_density, grid_density)

# 可視化（2x3のsubplotグリッド）
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. 真の関数（等高線）
contour1 = axes[0, 0].contour(
    X1_dense, X2_dense, true_dense, levels=15, colors="blue", alpha=0.8
)
axes[0, 0].clabel(contour1, inline=True, fontsize=8)
axes[0, 0].scatter(
    x_opt[0], x_opt[1], c="red", s=100, marker="*", label="Optimum", zorder=5
)
axes[0, 0].set_title("True Function f₁(x)")
axes[0, 0].set_xlabel("x₁")
axes[0, 0].set_ylabel("x₂")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. DeepGP予測（等高線）
contour2 = axes[0, 1].contour(
    X1_dense, X2_dense, dense_mean.numpy(), levels=15, colors="green", alpha=0.8
)
axes[0, 1].clabel(contour2, inline=True, fontsize=8)
axes[0, 1].scatter(
    x_opt[0], x_opt[1], c="red", s=100, marker="*", label="Optimum", zorder=5
)
axes[0, 1].set_title("DeepGP Prediction")
axes[0, 1].set_xlabel("x₁")
axes[0, 1].set_ylabel("x₂")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 予測誤差
error_dense = dense_mean.numpy() - true_dense
im1 = axes[0, 2].imshow(
    error_dense,
    extent=[input_range[0], input_range[1], input_range[0], input_range[1]],
    origin="lower",
    cmap="RdBu_r",
    alpha=0.8,
)
axes[0, 2].scatter(
    x_opt[0], x_opt[1], c="red", s=100, marker="*", label="Optimum", zorder=5
)
plt.colorbar(im1, ax=axes[0, 2], label="Prediction Error")
axes[0, 2].set_title("Prediction Error")
axes[0, 2].set_xlabel("x₁")
axes[0, 2].set_ylabel("x₂")
axes[0, 2].legend()

# 4. 予測不確実性（分散）
im2 = axes[1, 0].imshow(
    dense_var.numpy(),
    extent=[input_range[0], input_range[1], input_range[0], input_range[1]],
    origin="lower",
    cmap="plasma",
    alpha=0.8,
)
plt.colorbar(im2, ax=axes[1, 0], label="Prediction Variance")
axes[1, 0].set_title("Prediction Uncertainty")
axes[1, 0].set_xlabel("x₁")
axes[1, 0].set_ylabel("x₂")

# 5. 散布図：真の値 vs 予測値（テストデータ）
axes[1, 1].scatter(y_test.numpy(), test_mean.numpy(), alpha=0.6, s=20)
min_val = min(y_test.min().item(), test_mean.min().item())
max_val = max(y_test.max().item(), test_mean.max().item())
axes[1, 1].plot(
    [min_val, max_val], [min_val, max_val], "r--", alpha=0.8, label="Perfect Prediction"
)
axes[1, 1].set_xlabel("True Values")
axes[1, 1].set_ylabel("Predicted Values")
axes[1, 1].set_title(f"Prediction Accuracy (R² = {test_r2.item():.4f})")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. 残差プロット
residuals = (test_mean - y_test.squeeze()).numpy()
axes[1, 2].scatter(test_mean.numpy(), residuals, alpha=0.6, s=20)
axes[1, 2].axhline(y=0, color="r", linestyle="--", alpha=0.8)
axes[1, 2].set_xlabel("Predicted Values")
axes[1, 2].set_ylabel("Residuals")
axes[1, 2].set_title(f"Residual Plot (RMSE = {test_rmse.item():.4f})")
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 統計サマリー
print("=== DeepGP近似性能サマリー ===")
print(f"目標関数: f₁(x) = ||x - x_opt||² + f_opt")
print(f"  x_opt = {x_opt}")
print(f"  f_opt = {f_opt}")
print(f"学習データ数: {len(train_dataset)}")
print(f"テストRMSE: {test_rmse.item():.6f}")
print(f"テストR²: {test_r2.item():.6f}")
print(f"隠れ層次元数: {num_hidden_dims}")
print(f"隠れ層誘導点数: 16")
print(f"最終層誘導点数: 10")
print(f"学習時間: {training_time:.2f}秒")